# Fine-Tune DeepSeek-R1-Distill-Qwen-1.5B on Incident Response Dataset
This notebook fine-tunes the `DeepSeek-R1-Distill-Qwen-1.5B` model for binary classification (`malicious` vs `benign`) using your `fine_tuning_dataset_2.jsonl`. Training utilizes Apple MPS if available.


In [2]:
# Install dependencies (run once)
!pip install --upgrade pip
!pip install transformers datasets scikit-learn torch torchvision accelerate

In [3]:
# 1️⃣ Imports & Config
import os
import numpy as np
from datasets import load_dataset, disable_progress_bar
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Disable interactive progress bars
disable_progress_bar()

# Configuration
DATASET_JSONL = "fine_tuning_dataset_2.jsonl"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
OUTPUT_DIR = "./deepseek_qwen_ft"
NUM_LABELS = 2

# Device setup (MPS on Apple Silicon, else CUDA/CPU)
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [4]:
# 2️⃣ Load & Preprocess the Dataset
raw = load_dataset(
    "json",
    data_files={"train": DATASET_JSONL},
    split="train"
)
dataset = raw.train_test_split(test_size=0.1, seed=42)

def map_labels(example):
    lab = example["output"].strip().lower()
    return {"label": 1 if lab == "malicious" else 0}

dataset = dataset.map(map_labels)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
def tokenize_fn(examples):
    return tokenizer(examples["input"], truncation=True);

dataset = dataset.map(tokenize_fn, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Show summary
print(dataset)
print("\nExample entry:", dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'label', 'input_ids', 'attention_mask'],
        num_rows: 8725
    })
    test: Dataset({
        features: ['input', 'output', 'label', 'input_ids', 'attention_mask'],
        num_rows: 970
    })
})

Example entry: {'label': tensor(0), 'input_ids': tensor([151646,  20812,     25,    220,    760,  67684,   3298,     25,    220,
            19,     18,     17,     17,     23,     13,     15,    760,  67684,
          7084,     25,   2876,    220,     16,     18,    220,     16,     24,
            25,     17,     19,     25,     20,     20,   1217,  13013,  37328,
          1873,  29230,     67,     58,     17,     23,     17,     22,     19,
          5669,  13882,   1196,  84214,    504,    220,     16,     19,     17,
            13,     24,     18,     13,     16,     20,     22,     13,     17,
            18,     15,   2635,    220,     19,     18,     17,     17,     23,
           760,  14106,     25,  14

In [5]:
# 3️⃣ Define Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [6]:
# 4️⃣ Load Model & Setup Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, trust_remote_code=True, num_labels=NUM_LABELS
)
model.to(device)

# Ensure we have a pad token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=200,
    save_steps=200,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    bf16=False,
    dataloader_pin_memory=False,
    num_train_epochs=3,
    logging_dir="./logs_deepseek",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# 5️⃣ Train and Save
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model and tokenizer saved to {OUTPUT_DIR}")

Step,Training Loss,Validation Loss


RuntimeError: MPS backend out of memory (MPS allocated: 26.24 GB, other allocations: 19.13 GB, max allowed: 45.90 GB). Tried to allocate 890.25 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).